# Stage 1: Data Ingestion and Big Data Storage
**Goal:** download NYC TLC Yellow Taxi data, load it with PySpark, and store it in a partitioned Parquet data lake (raw and curated zones).

**Why Parquet + partitioning?** Parquet is columnar and compressed, so Spark reads only the columns it needs. Partitioning by month lets queries skip irrelevant files.

In [1]:
import os, requests
from pathlib import Path

BASE = Path('..') if Path('../data').exists() else Path('.')
RAW = BASE / 'data' / 'raw'
LAKE = BASE / 'data' / 'lake'
RAW.mkdir(parents=True, exist_ok=True)
LAKE.mkdir(parents=True, exist_ok=True)

YEAR = 2024
MONTHS = [1, 2, 3]   # change/add months to scale the dataset up
URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{y}-{m:02d}.parquet'
ZONES_URL = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

## 1. Download raw files

In [2]:
def download(url, dest):
    if dest.exists():
        print('exists:', dest.name); return
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    print('downloaded:', dest.name, round(dest.stat().st_size / 1e6, 1), 'MB')

for m in MONTHS:
    download(URL.format(y=YEAR, m=m), RAW / f'yellow_tripdata_{YEAR}-{m:02d}.parquet')
download(ZONES_URL, RAW / 'taxi_zone_lookup.csv')

downloaded: yellow_tripdata_2024-01.parquet 50.0 MB
downloaded: yellow_tripdata_2024-02.parquet 50.3 MB
downloaded: yellow_tripdata_2024-03.parquet 60.1 MB
downloaded: taxi_zone_lookup.csv 0.0 MB


## 2. Start Spark (local mode)

In [3]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName('nyc-taxi-stage1')
         .master('local[*]')
         .config('spark.driver.memory', '4g')
         .config('spark.sql.shuffle.partitions', '16')
         .getOrCreate())
print('Spark', spark.version)

Spark 4.0.4


## 3. Load raw data and inspect

In [4]:
raw_files = [str(RAW / f'yellow_tripdata_{YEAR}-{m:02d}.parquet') for m in MONTHS]
df = spark.read.parquet(*raw_files)
df.printSchema()
n_rows = df.count()
print('rows:', n_rows, '| columns:', len(df.columns))
df.show(5, truncate=False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

rows: 9554778 | columns: 19
+--------+--------------------+---------------------+---------------+-------------+----------+---

## 4. Dataset characteristics (for the report)

In [5]:
total_bytes = sum(os.path.getsize(f) for f in raw_files)
print('Raw size on disk (compressed Parquet): %.1f MB' % (total_bytes / 1e6))

nulls = df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).toPandas().T
nulls.columns = ['null_count']
nulls['null_pct'] = (nulls['null_count'] / n_rows * 100).round(2)
nulls

Raw size on disk (compressed Parquet): 160.4 MB


,null_count,null_pct
VendorID,0,0.00
tpep_pickup_datetime,0,0.00
tpep_dropoff_datetime,0,0.00
passenger_count,751962,7.87
trip_distance,0,0.00
RatecodeID,751962,7.87
store_and_fwd_flag,751962,7.87
PULocationID,0,0.00
DOLocationID,0,0.00
payment_type,0,0.00


## 5. Write the data lake
- **raw zone:** data as received, partitioned by pickup month.
- **curated zone:** created in Stage 2/3 after cleaning and feature engineering.

In [6]:
df_part = df.withColumn('pickup_month', F.date_format('tpep_pickup_datetime', 'yyyy-MM'))

raw_zone = str(LAKE / 'raw_zone' / 'yellow_trips')
(df_part.write.mode('overwrite')
        .partitionBy('pickup_month')
        .parquet(raw_zone))

zones = spark.read.csv(str(RAW / 'taxi_zone_lookup.csv'), header=True, inferSchema=True)
zones.write.mode('overwrite').parquet(str(LAKE / 'raw_zone' / 'taxi_zones'))
print('written to', raw_zone)

written to data/lake/raw_zone/yellow_trips


## 6. Verify partition pruning (shows why the lake design helps)

In [7]:
lake_df = spark.read.parquet(raw_zone)
q = lake_df.filter(F.col('pickup_month') == '2024-01')
q.explain()   # look for PartitionFilters in the plan
print('rows in 2024-01 partition:', q.count())
print(sorted(os.listdir(raw_zone)))

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [VendorID#223,tpep_pickup_datetime#224,tpep_dropoff_datetime#225,passenger_count#226L,trip_distance#227,RatecodeID#228L,store_and_fwd_flag#229,PULocationID#230,DOLocationID#231,payment_type#232L,fare_amount#233,extra#234,mta_tax#235,tip_amount#236,tolls_amount#237,improvement_surcharge#238,total_amount#239,congestion_surcharge#240,Airport_fee#241,pickup_month#242] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/data/lake/raw_zone/yellow_trips], PartitionFilters: [isnotnull(pickup_month#242), (pickup_month#242 = 2024-01)], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...


rows in 2024-01 partition: 2964617
['._SUCCESS.crc', '_SUCCESS', 'pickup_month=2002-12', 'pickup_month=2008-12', 'pickup_month=2009-01', 'pickup_month=2023-12', 'pickup_month=2024-01', 'pickup_month=2024-02', 'pickup_month=2024

## Notes for the report
- Record the row count, size, number of columns and null percentages printed above.
- Justify Spark (distributed processing, scales beyond RAM) and Parquet (columnar, compressed, partitioned) here.
- Next: Stage 2, data preparation.